# Setup

In [4]:
import pandas as pd
from datasets import load_dataset
from logging import getLogger, WARNING
from bs4 import BeautifulSoup

def load_glue_data(start_index: int, end_index: int):
  return load_dataset('glue', 'cola', split='validation') \
    .filter(lambda x: x['label'] == 1) \
    .remove_columns(['label']) \
    .select(range(start_index, end_index))

def load_toxic_data(start_index: int, end_index: int):
    # Load toxic comments data set
    getLogger('numexpr.utils').setLevel(WARNING)
    comments = pd.read_csv('../toxicity_annotated_comments.tsv', sep = '\t', index_col = 0)
    annotations = pd.read_csv('../toxicity_annotations.tsv',  sep = '\t')
    # labels a comment as toxic if the majority of annoatators did so
    labels = annotations.groupby('rev_id')['toxicity'].mean() > 0.5
    # join labels and comments
    comments['toxicity'] = labels
    # remove newline and tab tokens
    comments['comment'] = comments['comment'].apply(lambda x: x.replace("NEWLINE_TOKEN", " "))
    comments['comment'] = comments['comment'].apply(lambda x: x.replace("TAB_TOKEN", " "))
    test_comments = comments.query("split=='test'").query("toxicity==True")
    examples = test_comments.reset_index().to_dict('records')
    return examples[start_index:end_index]

def load_translation_data(start_index: int, end_index: int):
  # Build source and target mappings for BLEU scoring
  source = dict()
  target = dict()
  with open('../newstest2014-fren-src.en.sgm', 'r') as f:
    source_doc = BeautifulSoup(f, 'html.parser')
  with open('../newstest2014-fren-ref.fr.sgm', 'r') as f:
    target_doc = BeautifulSoup(f, 'html.parser')
  for doc in source_doc.find_all('doc'):
    source[str(doc['docid'])] = dict()
    for seg in doc.find_all('seg'):
      source[str(doc['docid'])][str(seg['id'])] = str(seg.string)
  for docid, doc in source.items():
    target[docid] = dict()
    for segid in doc:
      node = target_doc.select_one(f'doc[docid="{docid}"] > seg[id="{segid}"]')
      target[docid][segid] = str(node.string)
  # Sort the examples in order of length to improve runtime
  source_list = []
  for docid, doc in source.items():
    for segid, seg in doc.items():
      source_list.append({ docid: { segid: seg }})
  source_list.sort(key=lambda x: len(str(list(list(x.values())[0].values())[0])))
  source_list = source_list[start_index:end_index]
  output = []
  for example in source_list:
    for docid, doc in example.items():
      for segid, seg in doc.items():
        output.append({
            'docid': docid,
            'segid': segid,
            'english': seg,
            'french': target[docid][segid]
          })
  return output

def load_de_translation_data(start_index: int, end_index: int):
  # Build source and target mappings for BLEU scoring
  source = dict()
  target = dict()
  with open('../newstest2020-deen-src.de.sgm', 'r') as f:
    source_doc = BeautifulSoup(f, 'html.parser')
  with open('../newstest2020-deen-ref.en.sgm', 'r') as f:
    target_doc = BeautifulSoup(f, 'html.parser')
  for doc in source_doc.find_all('doc'):
    source[str(doc['docid'])] = dict()
    for seg in doc.find_all('seg'):
      source[str(doc['docid'])][str(seg['id'])] = str(seg.string)
  for docid, doc in source.items():
    target[docid] = dict()
    for segid in doc:
      node = target_doc.select_one(f'doc[docid="{docid}"] > p > seg[id="{segid}"]')
      target[docid][segid] = str(node.string)
  # Sort the examples in order of length to improve runtime
  source_list = []
  for docid, doc in source.items():
    for segid, seg in doc.items():
      source_list.append({ docid: { segid: seg }})
  source_list.sort(key=lambda x: len(str(list(list(x.values())[0].values())[0])))
  source_list = source_list[start_index:end_index]
  output = []
  for example in source_list:
    for docid, doc in example.items():
      for segid, seg in doc.items():
        output.append({
            'docid': docid,
            'segid': segid,
            'german': seg,
            'english': target[docid][segid]
          })
  return output

def load_squad_data(start_index: int, end_index: int):
  return load_dataset('squad', split='validation') \
    .select(range(start_index, end_index))

In [5]:
import ast
import numpy as np
from typing import Callable, List, Dict, Tuple
from torch import cuda, hub, no_grad, device as torchdevice
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer, AutoModelForQuestionAnswering
from fairseq import utils, tasks, checkpoint_utils, options
from collections import namedtuple
from fairseq.models.visual import VisualTextTransformerModel

def load_trocr(cpu: bool):
  # The "printed" models output all uppercase, so we use the "handwritten" models
  # which appear to perform well on computer-generated text as well
  device = 'cuda:0' if not cpu and cuda.is_available() else 'cpu'
  processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
  model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten').to(device)
  print(f"TrOCR configured to use device {device}.")
  device, processor, model
  def trocr(image):
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    generated_ids = model.generate(pixel_values)
    del pixel_values # remove from GPU
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
  return trocr

def load_visrep():
    return VisualTextTransformerModel.from_pretrained(
                checkpoint_file='/app/WMT_de-en/checkpoint_best.pt',
                target_dict='/app/WMT_de-en/dict.en.txt',
                target_spm='/app/WMT_de-en/spm.model',
                src='de',
                image_font_path='/app/visrep/fairseq/data/visual/fonts/NotoSans-Regular.ttf'
            ).eval()

def load_canine(cpu: bool) -> Tuple[str,any]:
  device = 'cuda:0' if not cpu and cuda.is_available() else 'cpu'
  tokenizer = AutoTokenizer.from_pretrained("Splend1dchan/canine-s-squad")
  model = AutoModelForQuestionAnswering.from_pretrained("Splend1dchan/canine-s-squad").to(device)
  def canine(question: str, context: str, id: str):
    inputs = tokenizer(question, context, return_tensors="pt").to(device)
    with no_grad():
      outputs = model(**inputs)
    answer_start_index = outputs.start_logits.argmax()
    answer_end_index = outputs.end_logits.argmax()
    predict_answer_tokens = inputs.input_ids[0, answer_start_index : answer_end_index + 1]
    prediction = tokenizer.decode(predict_answer_tokens, skip_special_tokens=True)
    return [ { 'prediction_text': prediction, 'id': id } ]
  return canine

In [6]:
from PIL import Image, ImageDraw, ImageFont

def load_font(font: str, font_size: int) -> None:
  _d = ImageDraw.Draw(Image.new("RGB", (0, 0), 'white'))
  font = ImageFont.truetype(font, font_size)    
  global draw
  def draw(text: str) -> Image:
    x, y = _d.textsize(text, font=font)
    img = Image.new("RGB", (x+40, y+20), 'white')
    d = ImageDraw.Draw(img)
    d.text((20, 5), text, font=font, fill=0)
    return img

# Load Models

In [7]:
load_font("/app/arialuni.ttf", 32)

In [8]:
trocr = load_trocr(True)
trocr(draw("hello world"))

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_211/2920216335.py:8: DeprecationWarning: textsize is deprecated and will be removed in Pillow 10 (2023-07-01). Use textbbox or textlength instead.
  x, y = _d.textsize(text, font=font)


TrOCR configured to use device cpu.


/usr/local/lib/python3.8/dist-packages/transformers/generation/utils.py:1259: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'hello world.'

In [ ]:
visrep = load_visrep()
visrep.translate('Ich bin ein robustes Model')

In [ ]:
canine = load_canine(True)
canine("What color is the sky?", "Roses are red. Grass is green. Skies are blue. Fire is red.", 0)

# Load Datasets

In [ ]:
glue = load_glue_data(0, 500)
toxic = load_toxic_data(0, 500)
translation = load_translation_data(0, 500)
de_translation = load_de_translation_data(0, 500)
squad = load_squad_data(0, 500)

# TextAttack

In [ ]:
glue = load_glue_data(0, 500)

In [21]:
from textattack.models.wrappers import ModelWrapper

class TrocrWrapper(ModelWrapper):
    def __init__(self, trocr):
        self.model = trocr

    def __call__(self, text_input_list):
        output = []
        for text_input in text_input_list:
            output.append(self.model(draw(text_input)))
        return output

In [22]:
import warnings
warnings.filterwarnings('ignore')

from textattack.datasets import HuggingFaceDataset
from textattack.attack_recipes import Seq2SickCheng2018BlackBox
from textattack import Attacker

wrapped_trocr = TrocrWrapper(trocr)
dataset = HuggingFaceDataset(glue, dataset_columns=(['sentence'],'sentence'))
attack = Seq2SickCheng2018BlackBox.build(wrapped_trocr)

attacker = Attacker(attack, dataset)
attacker.attack_dataset()

textattack: Unknown if model of class <class 'function'> compatible with goal function <class 'textattack.goal_functions.text.non_overlapping_output.NonOverlappingOutput'>.


Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  NonOverlappingOutput
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): LevenshteinEditDistance(
        (max_edit_distance):  30
        (compare_against_original):  True
      )
    (1): RepeatModification
    (2): StopwordModification
  (is_black_box):  True
) 






  0%|          | 0/10 [18:47<?, ?it/s]


KeyboardInterrupt: 

In [23]:
from toxic.core.model import ModelWrapper
getLogger().setLevel(WARNING)
toxic_model = ModelWrapper()

ModuleNotFoundError: No module named 'toxic'